# Collaborative Filtering Recommendation System

This notebook builds a user-based collaborative filtering system to recommend books based on user behavior.

The approach identifies similar users using cosine similarity and recommends books liked by those similar users.

This is part of the Goodreads Discovery Engine project.


In [1]:
# Setup libraries and tools for data processing and similarity computation
import pandas as pd  # data manipulation
import numpy as np  # numeric operations

from sklearn.preprocessing import LabelEncoder  # encode categorical labels when needed
from sklearn.metrics.pairwise import cosine_similarity  # compute similarity between users

## Data Loading (Optimized)

The original dataset is large, so we load a sampled subset using chunking.

This ensures faster execution and avoids memory issues on local systems while still preserving meaningful interaction patterns.


In [2]:
# Load a sampled subset of the interactions dataset for efficient local experimentation
chunks = []
chunk_size = 50000  # read data in chunks to avoid memory spikes
max_rows = 50000  # limit total rows for faster local development

for chunk in pd.read_json(
    "../data/goodreads_interactions_children.json.gz",
    lines=True,
    compression="gzip",
    chunksize=chunk_size
):
    chunks.append(chunk)
    if sum(len(c) for c in chunks) >= max_rows:
        break

interactions = pd.concat(chunks).sample(max_rows, random_state=42)
# combine chunks and sample the final dataset consistently
print("Loaded:", interactions.shape)

Loaded: (50000, 10)


In [3]:
# Filter interactions to keep active users and popular books
user_counts = interactions["user_id"].value_counts()  # count ratings per user
book_counts = interactions["book_id"].value_counts()  # count ratings per book

active_users = user_counts[user_counts >= 3].index  # keep users with at least 3 ratings
popular_books = book_counts[book_counts >= 5].index  # keep books rated by at least 5 users

filtered = interactions[
    (interactions["user_id"].isin(active_users)) &
    (interactions["book_id"].isin(popular_books)) &
    (interactions["rating"] > 0)
].copy()
# filter out noise and unrated interactions

In [4]:
# Apply a stricter filter to reduce noise in the training data
user_counts = interactions["user_id"].value_counts()  # recalc counts for a stricter filter
book_counts = interactions["book_id"].value_counts()  # recalc count per book

active_users = user_counts[user_counts >= 5].index   # users with at least 5 ratings
popular_books = book_counts[book_counts >= 10].index  # books with at least 10 ratings

filtered_interactions = interactions[
    (interactions["user_id"].isin(active_users)) &
    (interactions["book_id"].isin(popular_books)) &
    (interactions["rating"] > 0)
].copy()
# use a smaller, cleaner dataset for model building

print("After filtering:", filtered_interactions.shape)

After filtering: (16953, 10)


In [5]:
# Focus on the most active users and most popular books for the model
top_users = filtered["user_id"].value_counts().head(300).index  # choose top 300 active users
top_books = filtered["book_id"].value_counts().head(300).index  # choose top 300 popular books

filtered = filtered[
    (filtered["user_id"].isin(top_users)) &
    (filtered["book_id"].isin(top_books))
].copy()
# narrow dataset to the most active users and most rated books

print(filtered.shape)

(9321, 10)


In [6]:
# Build the user-book rating matrix as a pivot table
pivot = filtered.pivot_table(
    index="user_id",
    columns="book_id",
    values="rating",
    fill_value=0
)

print(pivot.shape)

(300, 300)


In [7]:
# Compute user similarity using cosine similarity on the pivot matrix
user_similarity = cosine_similarity(pivot)

print("Similarity matrix shape:", user_similarity.shape)

Similarity matrix shape: (300, 300)


In [8]:
# Define recommendation logic based on similar users' liked books
def recommend_books(user_id, n=10):
    # find index for the target user in the pivot table
    user_index = pivot.index.get_loc(user_id)
    
    # get similarity scores for the target user against all other users
    sim_scores = list(enumerate(user_similarity[user_index]))
    # sort scores and remove the user itself from the top matches
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:6]
    
    similar_users = [i[0] for i in sim_scores]  # top 5 similar users
    
    recommended_books = {}
    
    for u in similar_users:
        books_seen = pivot.iloc[u]
        liked_books = books_seen[books_seen > 0].index  # books rated by the similar user
        
        for book in liked_books:
            recommended_books[book] = recommended_books.get(book, 0) + 1
    
    # sort books by how often they appear among similar users
    sorted_books = sorted(recommended_books.items(), key=lambda x: x[1], reverse=True)
    
    results = []
    for book_id, score in sorted_books[:n]:
        title = book_id_to_title.get(book_id, "Unknown")
        results.append((book_id, title, score))
    
    return results


In [9]:
# Run a quick recommendation test for the first user in the pivot table
user = pivot.index[0]  # test with the first user in the pivot table
print("Recommendations:", recommend_books(user))

NameError: name 'book_id_to_title' is not defined

In [ ]:
# Load book metadata so recommendations can include readable titles
books = pd.read_json(
    "../data/goodreads_books_children.json.gz",
    lines=True,
    compression="gzip"
)
# load book metadata for title lookup

print(books.shape)

(124082, 29)


In [ ]:
# Create a mapping from book IDs to titles for recommendation output
book_id_to_title = dict(zip(books["book_id"], books["title"]))  # create fast lookup from book id to title

In [ ]:
# Print the recommended books for the selected user
user = pivot.index[0]  # reuse the first user for display
print("Recommendations:")
for rec in recommend_books(user):
    print(rec)

Recommendations:
(24178, "Charlotte's Web", 5)
(30119, 'Where the Sidewalk Ends', 5)
(32929, 'Goodnight Moon', 5)
(5, 'Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)', 4)
(2998, 'The Secret Garden', 4)
(3636, 'The Giver (The Giver, #1)', 4)
(6310, 'Charlie and the Chocolate Factory (Charlie Bucket, #1)', 4)
(7784, 'The Lorax', 4)
(19321, 'The Tale of Peter Rabbit', 4)
(23772, 'Green Eggs and Ham', 4)
